# CookSmart: Hybrid Recipe Recommendation System

**Author:** Max Chiu  
**Goal:** Recommend recipes a user is likely to enjoy by combining historical user interactions with recipe content.

## 1. Introduction

This notebook reframes the original recipe-rating project as a personalized recommendation problem. Instead of predicting one global average rating for every recipe, the system predicts which unseen recipes a specific user is likely to like.

Three approaches are compared:

1. **Popularity Baseline** — recommends recipes frequently liked in the training data.
2. **Content-based Baseline** — recommends recipes similar to a user's previously liked recipes using text features.
3. **Weighted Hybrid Recommender** — combines popularity and content similarity, with the weight selected on a validation split.

The final evaluation uses a sampled-ranking setting: one held-out recipe the user liked plus 99 unseen candidate recipes. Metrics are **Recall@10** and **NDCG@10**.

## 2. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import ast
from collections import defaultdict
import math
import seaborn as sns
import matplotlib.pyplot as plt

import plotly.express as px
# pd.options.plotting.backend = 'plotly'

# from dsc80_utils import * # Feel free to uncomment and use this.

## 3. Data Cleaning and Feature Preparation

The recipe table is cleaned at the recipe level. Structured nutrition values are expanded into numeric columns, while tags and ingredients are parsed for later content-based modeling.

The interaction table is kept separately because recommendation depends on the relationship between a user and a recipe. Ratings of 4 or 5 are treated as positive feedback; rating 0 is not treated as a negative preference.

In [ ]:
recipes_path = "/Users/maxchiu/Desktop/dsc80-2024-fa/projects/project04/RAW_recipes.csv"
interactions_path = "/Users/maxchiu/Desktop/dsc80-2024-fa/projects/project04/RAW_interactions.csv"

recipes_raw = pd.read_csv(recipes_path)
interactions_raw = pd.read_csv(interactions_path)


def safe_parse_list(value):
    """將 CSV 中長得像 list 的字串轉成 Python list。"""
    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []


# -------------------------------------------------
# 1. 食譜資料：每道食譜只保留一列
# -------------------------------------------------
recipes_clean = recipes_raw.copy()

# 日期欄位
recipes_clean["submitted"] = pd.to_datetime(
    recipes_clean["submitted"],
    errors="coerce"
)

# 保留 description 是否缺失，文字本身補成空字串
recipes_clean["description_missing"] = recipes_clean["description"].isna().astype(int)
recipes_clean["name"] = recipes_clean["name"].fillna("")
recipes_clean["description"] = recipes_clean["description"].fillna("")

# 將原本字串格式的欄位轉成 list
for col in ["tags", "ingredients", "steps", "nutrition"]:
    recipes_clean[col] = recipes_clean[col].apply(safe_parse_list)

# nutrition 拆成數值特徵
nutrition_columns = [
    "calories",
    "total_fat",
    "sugar",
    "sodium",
    "protein",
    "saturated_fat",
    "carbohydrates",
]

nutrition_df = pd.DataFrame(
    recipes_clean["nutrition"].tolist(),
    columns=nutrition_columns,
    index=recipes_clean.index,
)

recipes_clean = pd.concat(
    [recipes_clean.drop(columns="nutrition"), nutrition_df],
    axis=1,
)

# 確保營養欄位是數值
recipes_clean[nutrition_columns] = recipes_clean[nutrition_columns].apply(
    pd.to_numeric,
    errors="coerce",
)

# 建立後續做內容推薦與 embedding 時使用的文字欄位
recipes_clean["tags_text"] = recipes_clean["tags"].apply(" ".join)
recipes_clean["ingredients_text"] = recipes_clean["ingredients"].apply(" ".join)
recipes_clean["steps_text"] = recipes_clean["steps"].apply(" ".join)

recipes_clean["recipe_text"] = (
    recipes_clean["name"] + " "
    + recipes_clean["description"] + " "
    + recipes_clean["tags_text"] + " "
    + recipes_clean["ingredients_text"]
).str.lower()

# 移除沒有有效 recipe id 的重複資料
recipes_clean = (
    recipes_clean
    .dropna(subset=["id"])
    .drop_duplicates(subset=["id"])
    .reset_index(drop=True)
)

# 目前不需要 average_rating，避免未來資料洩漏
print("recipes_clean shape:", recipes_clean.shape)
recipes_clean.head()


In [ ]:
# -------------------------------------------------
# 2. 使用者互動資料：保留每一筆 user-recipe interaction
# -------------------------------------------------
interactions_clean = interactions_raw.copy()

interactions_clean["date"] = pd.to_datetime(
    interactions_clean["date"],
    errors="coerce"
)

# rating = 0 視為沒有有效明確評分，不當作負評
interactions_clean = interactions_clean[
    interactions_clean["rating"].between(1, 5)
].copy()

# 推薦任務的正向互動：4 或 5 分
interactions_clean["liked"] = (
    interactions_clean["rating"] >= 4
).astype(int)

# 只保留有對應食譜內容的互動
interactions_clean = interactions_clean[
    interactions_clean["recipe_id"].isin(recipes_clean["id"])
].copy()

# 確保同一使用者對同一食譜沒有重複互動
interactions_clean = interactions_clean.drop_duplicates(
    subset=["user_id", "recipe_id"],
    keep="last",
).reset_index(drop=True)

print("interactions_clean shape:", interactions_clean.shape)
print(interactions_clean["rating"].value_counts().sort_index())
interactions_clean

In [ ]:
data_overview = pd.DataFrame({
    'Dataset': ['Recipes', 'Valid user–recipe interactions', 'Positive interactions (rating ≥ 4)'],
    'Rows': [
        len(recipes_clean),
        len(interactions_clean),
        int(interactions_clean['liked'].sum())
    ]
})

display(data_overview)
print('Recipe date range:', recipes_clean['submitted'].min().date(), 'to', recipes_clean['submitted'].max().date())
print('Interaction date range:', interactions_clean['date'].min().date(), 'to', interactions_clean['date'].max().date())

## 4. Time-based Train/Test Split

For each user with at least two positive interactions, the most recent liked recipe is held out for testing. Earlier liked recipes form the training history. This simulates recommending a future recipe from past preferences and avoids using the held-out interaction during training.

In [ ]:
# -------------------------------------------------
# 3. 時間切分：用過去喜好預測使用者未來會喜歡什麼
# -------------------------------------------------

# 只使用正向互動作為推薦模型的偏好訊號
positive_interactions = (
    interactions_clean[interactions_clean["liked"] == 1]
    .sort_values(["user_id", "date", "recipe_id"])
    .copy()
)

# 只保留至少喜歡過 2 道食譜的使用者
# 一道留給測試，至少還有一道可用來建立偏好
user_positive_counts = positive_interactions.groupby("user_id").size()

eligible_users = user_positive_counts[
    user_positive_counts >= 2
].index

positive_interactions = positive_interactions[
    positive_interactions["user_id"].isin(eligible_users)
].copy()

# 每位使用者最後一次喜歡的食譜 = test set
test_interactions = (
    positive_interactions
    .groupby("user_id", group_keys=False)
    .tail(1)
    .copy()
)

# 其餘較早的喜歡紀錄 = train set
test_index = test_interactions.index

train_interactions = (
    positive_interactions
    .drop(index=test_index)
    .copy()
)

# 方便後續使用的 user / recipe 集合
train_users = train_interactions["user_id"].unique()
train_recipe_ids = train_interactions["recipe_id"].unique()

print("Train positive interactions:", len(train_interactions))
print("Test positive interactions:", len(test_interactions))
print("Users in evaluation:", test_interactions["user_id"].nunique())

# 檢查每個測試使用者都仍至少有一筆過去喜好可供推薦
assert train_interactions["user_id"].nunique() == test_interactions["user_id"].nunique()

# 檢查同一筆 user-recipe 互動沒有同時出現在 train 與 test
train_pairs = set(
    zip(train_interactions["user_id"], train_interactions["recipe_id"])
)
test_pairs = set(
    zip(test_interactions["user_id"], test_interactions["recipe_id"])
)

assert len(train_pairs.intersection(test_pairs)) == 0

print("Time split completed.")

## 5. Popularity Baseline

This baseline recommends recipes with the highest number of positive interactions in the training set, while excluding recipes already seen by the target user. It establishes how well a non-personalized strategy performs.

In [ ]:
K = 10

# -------------------------------------------------
# Popularity Baseline只使用 train_interactions 計算食譜熱門程度
# -------------------------------------------------

# 訓練期間被使用者喜歡的次數
recipe_popularity = (
    train_interactions
    .groupby("recipe_id")
    .size()
    .sort_values(ascending=False)
)

popular_recipe_ids = recipe_popularity.index.tolist()

print("Number of recipes with positive train interactions:", len(popular_recipe_ids))
print(recipe_popularity.head(10))

In [ ]:
# -------------------------------------------------
# 5. 建立每位使用者在測試前已互動過的食譜清單
# -------------------------------------------------

test_dates = test_interactions[
    ["user_id", "date"]
].rename(columns={"date": "test_date"})

# 只保留該使用者在測試答案之前的互動
history_before_test = interactions_clean.merge(
    test_dates,
    on="user_id",
    how="inner"
)

history_before_test = history_before_test[
    history_before_test["date"] < history_before_test["test_date"]
].copy()

seen_recipes_by_user = (
    history_before_test
    .groupby("user_id")["recipe_id"]
    .apply(set)
    .to_dict()
)

In [ ]:
# -------------------------------------------------
# 6. Popularity recommendation function
# -------------------------------------------------

def recommend_popular_recipes(user_id, k=10):
    seen_recipes = seen_recipes_by_user.get(user_id, set())

    recommendations = []

    for recipe_id in popular_recipe_ids:
        if recipe_id not in seen_recipes:
            recommendations.append(recipe_id)

        if len(recommendations) == k:
            break

    return recommendations

# -------------------------------------------------
# 7. 評估 Popularity Baseline
# -------------------------------------------------

def evaluate_recommender(test_df, k=10):
    hits = []
    ndcg_scores = []

    for row in test_df.itertuples(index=False):
        user_id = row.user_id
        actual_recipe_id = row.recipe_id

        recommendations = recommend_popular_recipes(user_id, k=k)

        if actual_recipe_id in recommendations:
            rank = recommendations.index(actual_recipe_id) + 1

            # 一個 test 食譜有被推薦到，就算 hit
            hits.append(1)

            # 排越前面分數越高
            ndcg_scores.append(1 / math.log2(rank + 1))
        else:
            hits.append(0)
            ndcg_scores.append(0)

    return {
        f"Recall@{k}": np.mean(hits),
        f"NDCG@{k}": np.mean(ndcg_scores),
        "evaluated_users": len(test_df),
    }


popularity_results = evaluate_recommender(
    test_interactions,
    k=K
)

print("Popularity Baseline Results")
print(popularity_results)

In [ ]:
top_popular = (
    recipe_popularity.head(10).rename('positive_likes').reset_index()
    .merge(recipes_clean[['id', 'name']], left_on='recipe_id', right_on='id', how='left')
)

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=top_popular.sort_values('positive_likes'),
    x='positive_likes', y='name', color='#4C78A8'
)
plt.title('Top 10 Popular Recipes in the Training Set', weight='bold')
plt.xlabel('Positive interactions (rating ≥ 4)')
plt.ylabel('Recipe name')
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', padding=3)
plt.tight_layout()
plt.show()

## 6. Content-based Baseline

Recipe names, descriptions, tags, and ingredients are transformed with TF-IDF. A user's content profile is the normalized aggregate of recipes they liked during training. Candidate recipes are ranked by cosine similarity to that profile.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix

recipes_model = (
    recipes_clean[["id", "name", "recipe_text"]]
    .drop_duplicates(subset="id")
    .reset_index(drop=True)
)

# 食譜 id 與 TF-IDF 矩陣列數的對照
recipe_id_to_idx = {
    recipe_id: idx
    for idx, recipe_id in enumerate(recipes_model["id"])
}

all_recipe_ids = recipes_model["id"].to_numpy()

# 將名稱、描述、標籤、食材轉成 TF-IDF 特徵
tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_features=30_000,
    sublinear_tf=True
)

recipe_tfidf = tfidf.fit_transform(
    recipes_model["recipe_text"].fillna("")
)

print("TF-IDF matrix shape:", recipe_tfidf.shape)

In [ ]:
# -------------------------------------------------
# 建立每位使用者過去喜歡的食譜清單
# -------------------------------------------------

train_liked_recipes_by_user = (
    train_interactions
    .groupby("user_id")["recipe_id"]
    .apply(list)
    .to_dict()
)


def get_user_profile(user_id):
    """
    將使用者過去喜歡的食譜 TF-IDF 向量平均，
    作為此使用者的內容偏好輪廓。
    """
    liked_recipe_ids = train_liked_recipes_by_user.get(user_id, [])

    liked_indices = [
        recipe_id_to_idx[recipe_id]
        for recipe_id in liked_recipe_ids
        if recipe_id in recipe_id_to_idx
    ]

    if not liked_indices:
        return None

    # 平均使用者喜歡食譜的內容向量
    profile = recipe_tfidf[liked_indices].sum(axis=0)
    profile = csr_matrix(profile)

    # 正規化後，內積可視為 cosine similarity
    return normalize(profile)

# -------------------------------------------------
# Content-based recommendation function
# -------------------------------------------------

def recommend_content_recipes(user_id, k=10):
    user_profile = get_user_profile(user_id)

    if user_profile is None:
        return []

    # 排除使用者在測試答案前已經互動過的食譜
    seen_recipe_ids = seen_recipes_by_user.get(user_id, set())

    candidate_mask = ~np.isin(
        all_recipe_ids,
        list(seen_recipe_ids)
    )

    candidate_indices = np.where(candidate_mask)[0]

    # 使用者偏好與每道候選食譜的 cosine similarity
    scores = (
        user_profile
        @ recipe_tfidf[candidate_indices].T
    ).toarray().ravel()

    # 取分數最高的前 k 名
    top_local_indices = np.argsort(scores)[-k:][::-1]
    top_recipe_indices = candidate_indices[top_local_indices]

    return all_recipe_ids[top_recipe_indices].tolist()

In [ ]:
# -------------------------------------------------
# 在相同候選集上比較 Popularity 與 Content-based
# -------------------------------------------------

rng = np.random.default_rng(42)

# 對齊每道食譜的 popularity score；未被喜歡過則為 0
popularity_score_array = (
    pd.Series(all_recipe_ids)
    .map(recipe_popularity)
    .fillna(0)
    .to_numpy()
)


def get_candidate_set(user_id, actual_recipe_id, n_negatives=99):
    """回傳：真實喜歡食譜 + 隨機未互動食譜。"""
    seen_recipe_ids = seen_recipes_by_user.get(user_id, set())

    valid_mask = ~np.isin(
        all_recipe_ids,
        list(seen_recipe_ids | {actual_recipe_id})
    )

    negative_pool = all_recipe_ids[valid_mask]

    negative_recipe_ids = rng.choice(
        negative_pool,
        size=n_negatives,
        replace=False
    )

    candidates = np.concatenate(
        [[actual_recipe_id], negative_recipe_ids]
    )

    # 打亂，避免同分數時測試答案永遠在固定位置
    return rng.permutation(candidates)


def get_rank(actual_recipe_id, candidate_recipe_ids, scores):
    """回傳實際喜歡食譜在候選集中的名次。"""
    actual_position = np.where(
        candidate_recipe_ids == actual_recipe_id
    )[0][0]

    ranking = np.argsort(-scores, kind="stable")
    return np.where(ranking == actual_position)[0][0] + 1


def calculate_ranking_metrics(ranks, k=10):
    ranks = np.array(ranks)

    return {
        f"Recall@{k}": np.mean(ranks <= k),
        f"NDCG@{k}": np.mean(
            np.where(
                ranks <= k,
                1 / np.log2(ranks + 1),
                0
            )
        ),
        "evaluated_users": len(ranks),
    }


popularity_ranks = []
content_ranks = []

for row in test_interactions.itertuples(index=False):
    user_id = row.user_id
    actual_recipe_id = row.recipe_id

    candidate_recipe_ids = get_candidate_set(
        user_id,
        actual_recipe_id,
        n_negatives=99
    )

    candidate_indices = np.array([
        recipe_id_to_idx[recipe_id]
        for recipe_id in candidate_recipe_ids
    ])

    # Popularity 的分數
    popularity_scores = popularity_score_array[candidate_indices]

    popularity_ranks.append(
        get_rank(
            actual_recipe_id,
            candidate_recipe_ids,
            popularity_scores
        )
    )

    # Content-based 的分數
    user_profile = get_user_profile(user_id)

    content_scores = (
        user_profile
        @ recipe_tfidf[candidate_indices].T
    ).toarray().ravel()

    content_ranks.append(
        get_rank(
            actual_recipe_id,
            candidate_recipe_ids,
            content_scores
        )
    )

content_results = calculate_ranking_metrics(content_ranks, k=10)

popularity_sampled_results = calculate_ranking_metrics(
    popularity_ranks,
    k=10
)

print("Popularity Baseline — sampled candidates")
print(popularity_sampled_results)

print("\nContent-based Baseline — sampled candidates")
print(content_results)

## 7. LightFM Experiment

LightFM is included as an additional hybrid-model experiment. It learns from the user–recipe interaction matrix and recipe tags/ingredients. Its result is reported for transparency, but the weighted hybrid below is selected as the final model only if it performs best on the held-out test set.

In [ ]:
from lightfm import LightFM
from lightfm.data import Dataset

In [ ]:
# LightFM 的 user / item ID 使用字串，避免型別問題
lightfm_train = train_interactions.copy()
lightfm_train["user_id_str"] = lightfm_train["user_id"].astype(str)
lightfm_train["recipe_id_str"] = lightfm_train["recipe_id"].astype(str)

recipes_lightfm = recipes_clean[
    ["id", "tags", "ingredients"]
].copy()

recipes_lightfm["recipe_id_str"] = recipes_lightfm["id"].astype(str)


def make_item_features(row):
    """
    將食譜的 tags 與 ingredients 轉成 LightFM item features。
    加上 tag: / ingredient: 前綴，避免相同文字被誤認為同一種特徵。
    """
    features = []

    for tag in row["tags"]:
        features.append(f"tag:{tag}")

    for ingredient in row["ingredients"]:
        features.append(f"ingredient:{ingredient}")

    # 避免食譜沒有任何特徵
    if not features:
        features.append("meta:unknown")

    return list(set(features))


recipes_lightfm["item_features"] = recipes_lightfm.apply(
    make_item_features,
    axis=1
)

all_item_feature_tokens = sorted(
    {
        feature
        for feature_list in recipes_lightfm["item_features"]
        for feature in feature_list
    }
)

# 建立 LightFM 的 user、item 與 item feature mapping
dataset = Dataset()

dataset.fit(
    users=lightfm_train["user_id_str"].unique(),
    items=recipes_lightfm["recipe_id_str"].unique(),
    item_features=all_item_feature_tokens
)

# 建立使用者—食譜正向互動矩陣
interactions_matrix, _ = dataset.build_interactions(
    (
        row.user_id_str,
        row.recipe_id_str
    )
    for row in lightfm_train.itertuples(index=False)
)

# 建立食譜特徵矩陣
item_features_matrix = dataset.build_item_features(
    (
        row.recipe_id_str,
        row.item_features
    )
    for row in recipes_lightfm.itertuples(index=False)
)

print("Interaction matrix:", interactions_matrix.shape)
print("Item feature matrix:", item_features_matrix.shape)

In [ ]:
# WARP 適合「把使用者可能喜歡的項目排到前面」的推薦任務
lightfm_model = LightFM(
    loss="warp",
    no_components=64,
    learning_rate=0.05,
    random_state=42
)

lightfm_model.fit(
    interactions_matrix,
    item_features=item_features_matrix,
    epochs=15,
    num_threads=4,
    verbose=True
)

In [ ]:
# Retrieve LightFM's internal ID mappings for prediction and evaluation
(
    user_id_map,
    user_feature_map,
    item_id_map,
    item_feature_map
) = dataset.mapping()

print(f"LightFM users: {len(user_id_map):,}")
print(f"LightFM recipes: {len(item_id_map):,}")
print(f"LightFM item features: {len(item_feature_map):,}")


In [ ]:


K = 10
rng = np.random.default_rng(42)

# 建立每位測試使用者在測試答案前看過的食譜
test_dates = test_interactions[
    ["user_id", "date"]
].rename(columns={"date": "test_date"})

history_before_test = interactions_clean.merge(
    test_dates,
    on="user_id",
    how="inner"
)

history_before_test = history_before_test[
    history_before_test["date"] < history_before_test["test_date"]
]

test_seen_recipes_by_user = (
    history_before_test
    .groupby("user_id")["recipe_id"]
    .apply(set)
    .to_dict()
)


def calculate_metrics(ranks, k=10):
    ranks = np.array(ranks)

    recall = np.mean(ranks <= k)

    ndcg = np.mean(
        np.where(
            ranks <= k,
            1 / np.log2(ranks + 1),
            0
        )
    )

    return {
        f"Recall@{k}": recall,
        f"NDCG@{k}": ndcg,
        "evaluated_users": len(ranks)
    }


lightfm_ranks = []

for row in test_interactions.itertuples(index=False):
    user_id = row.user_id
    actual_recipe_id = row.recipe_id

    # 該使用者在測試前已經互動過的食譜不能再推薦
    seen_recipes = test_seen_recipes_by_user.get(user_id, set())

    candidate_mask = ~np.isin(
        all_recipe_ids,
        list(seen_recipes | {actual_recipe_id})
    )

    negative_pool = all_recipe_ids[candidate_mask]

    # 真實喜歡食譜 + 99 道隨機未互動食譜
    negative_recipe_ids = rng.choice(
        negative_pool,
        size=99,
        replace=False
    )

    candidate_recipe_ids = np.concatenate(
        [[actual_recipe_id], negative_recipe_ids]
    )

    # 打亂候選順序，避免同分時固定位置影響結果
    candidate_recipe_ids = rng.permutation(candidate_recipe_ids)

    # 轉為 LightFM 內部 index
    user_index = user_id_map[str(user_id)]

    candidate_item_indices = np.array([
        item_id_map[str(recipe_id)]
        for recipe_id in candidate_recipe_ids
    ])

    # LightFM 對候選食譜給分
    scores = lightfm_model.predict(
        user_ids=np.repeat(user_index, len(candidate_item_indices)),
        item_ids=candidate_item_indices,
        item_features=item_features_matrix,
        num_threads=4
    )

    # 找出真實喜歡食譜的排名
    ranking = np.argsort(-scores, kind="stable")

    actual_position = np.where(
        candidate_recipe_ids == actual_recipe_id
    )[0][0]

    actual_rank = np.where(
        ranking == actual_position
    )[0][0] + 1

    lightfm_ranks.append(actual_rank)


lightfm_results = calculate_metrics(
    lightfm_ranks,
    k=K
)

print("LightFM Hybrid Recommender Results")
print(lightfm_results)

## 8. Weighted Hybrid Recommender

The final recommender combines two complementary signals:

$$\text{Hybrid score} = \alpha \times \text{content similarity} + (1 - \alpha) \times \text{popularity score}$$

A validation split from the training interactions selects the best value of $\alpha$. The selected weight is then evaluated once on the untouched test set.

In [ ]:
# -------------------------------------------------
# 12. 從 train 再切出 validation，選 Hybrid 權重
# -------------------------------------------------

# 只有至少還有兩筆 train 正向互動的使用者，
# 才能留最後一筆當 validation，並保留更早紀錄建立偏好
train_counts = train_interactions.groupby("user_id").size()

validation_users = train_counts[
    train_counts >= 2
].index

validation_interactions = (
    train_interactions[
        train_interactions["user_id"].isin(validation_users)
    ]
    .sort_values(["user_id", "date", "recipe_id"])
    .groupby("user_id", group_keys=False)
    .tail(1)
    .copy()
)

fit_interactions = train_interactions.drop(
    index=validation_interactions.index
).copy()

print("Fit interactions:", len(fit_interactions))
print("Validation interactions:", len(validation_interactions))

In [ ]:
def build_seen_recipes(interactions_df, eval_df):
    eval_dates = eval_df[
        ["user_id", "date"]
    ].rename(columns={"date": "eval_date"})

    history = interactions_df.merge(
        eval_dates,
        on="user_id",
        how="inner"
    )

    history = history[
        history["date"] < history["eval_date"]
    ]

    return (
        history
        .groupby("user_id")["recipe_id"]
        .apply(set)
        .to_dict()
    )

def build_liked_recipes_by_user(interactions_df):
    return (
        interactions_df
        .groupby("user_id")["recipe_id"]
        .apply(list)
        .to_dict()
    )


def get_user_profile_from_history(user_id, liked_recipes_by_user):
    liked_recipe_ids = liked_recipes_by_user.get(user_id, [])

    liked_indices = [
        recipe_id_to_idx[recipe_id]
        for recipe_id in liked_recipe_ids
        if recipe_id in recipe_id_to_idx
    ]

    if not liked_indices:
        return None

    profile = recipe_tfidf[liked_indices].sum(axis=0)
    return normalize(csr_matrix(profile))


def min_max_normalize(scores):
    scores = np.asarray(scores, dtype=float)

    if scores.max() == scores.min():
        return np.zeros_like(scores)

    return (scores - scores.min()) / (
        scores.max() - scores.min()
    )


def build_score_cache(
    eval_df,
    liked_recipes_by_user,
    seen_recipes,
    popularity_series,
    n_negatives=99,
    seed=42
):
    rng = np.random.default_rng(seed)

    popularity_array = (
        pd.Series(all_recipe_ids)
        .map(popularity_series)
        .fillna(0)
        .to_numpy()
    )

    score_cache = []

    for row in eval_df.itertuples(index=False):
        user_id = row.user_id
        actual_recipe_id = row.recipe_id

        seen = seen_recipes.get(user_id, set())

        candidate_mask = ~np.isin(
            all_recipe_ids,
            list(seen | {actual_recipe_id})
        )

        negative_pool = all_recipe_ids[candidate_mask]

        negative_recipe_ids = rng.choice(
            negative_pool,
            size=n_negatives,
            replace=False
        )

        candidate_recipe_ids = np.concatenate(
            [[actual_recipe_id], negative_recipe_ids]
        )

        candidate_recipe_ids = rng.permutation(candidate_recipe_ids)

        candidate_indices = np.array([
            recipe_id_to_idx[recipe_id]
            for recipe_id in candidate_recipe_ids
        ])

        user_profile = get_user_profile_from_history(
            user_id,
            liked_recipes_by_user
        )

        content_scores = (
            user_profile
            @ recipe_tfidf[candidate_indices].T
        ).toarray().ravel()

        popularity_scores = popularity_array[candidate_indices]

        score_cache.append({
            "actual_recipe_id": actual_recipe_id,
            "candidate_recipe_ids": candidate_recipe_ids,
            "content_scores": min_max_normalize(content_scores),
            "popularity_scores": min_max_normalize(popularity_scores),
        })

    return score_cache


def evaluate_hybrid(score_cache, content_weight, k=10):
    ranks = []

    for item in score_cache:
        hybrid_scores = (
            content_weight * item["content_scores"]
            + (1 - content_weight) * item["popularity_scores"]
        )

        ranking = np.argsort(-hybrid_scores, kind="stable")

        actual_position = np.where(
            item["candidate_recipe_ids"] == item["actual_recipe_id"]
        )[0][0]

        rank = np.where(ranking == actual_position)[0][0] + 1
        ranks.append(rank)

    return calculate_ranking_metrics(ranks, k=k)

In [ ]:
# -------------------------------------------------
# 13. 使用 validation 找最佳 content weight
# -------------------------------------------------

fit_liked_by_user = build_liked_recipes_by_user(
    fit_interactions
)

validation_seen_recipes = build_seen_recipes(
    interactions_clean,
    validation_interactions
)

fit_popularity = (
    fit_interactions
    .groupby("recipe_id")
    .size()
    .sort_values(ascending=False)
)

validation_score_cache = build_score_cache(
    eval_df=validation_interactions,
    liked_recipes_by_user=fit_liked_by_user,
    seen_recipes=validation_seen_recipes,
    popularity_series=fit_popularity,
    seed=42
)

weight_results = []

for content_weight in np.arange(0, 1.01, 0.1):
    metrics = evaluate_hybrid(
        validation_score_cache,
        content_weight=content_weight,
        k=10
    )

    weight_results.append({
        "content_weight": round(content_weight, 1),
        "popularity_weight": round(1 - content_weight, 1),
        **metrics
    })

weight_results_df = pd.DataFrame(weight_results)

display(weight_results_df)

best_row = weight_results_df.loc[
    weight_results_df["NDCG@10"].idxmax()
]

best_content_weight = best_row["content_weight"]

print("Best content weight:", best_content_weight)
print("Best validation result:")
print(best_row)

In [ ]:
# -------------------------------------------------
# 14. Final test evaluation
# -------------------------------------------------

test_liked_by_user = build_liked_recipes_by_user(
    train_interactions
)

test_seen_recipes = build_seen_recipes(
    interactions_clean,
    test_interactions
)

test_popularity = (
    train_interactions
    .groupby("recipe_id")
    .size()
    .sort_values(ascending=False)
)

test_score_cache = build_score_cache(
    eval_df=test_interactions,
    liked_recipes_by_user=test_liked_by_user,
    seen_recipes=test_seen_recipes,
    popularity_series=test_popularity,
    seed=99
)

final_results = pd.DataFrame([
    {
        "Model": "Popularity Baseline",
        **evaluate_hybrid(
            test_score_cache,
            content_weight=0,
            k=10
        )
    },
    {
        "Model": "Content-based Baseline",
        **evaluate_hybrid(
            test_score_cache,
            content_weight=1,
            k=10
        )
    },
    {
        "Model": f"Hybrid (content={best_content_weight:.1f})",
        **evaluate_hybrid(
            test_score_cache,
            content_weight=best_content_weight,
            k=10
        )
    }
])

display(final_results)

In [ ]:
# Evaluate LightFM on the exact same candidate sets used by final_results
def evaluate_lightfm_on_cached_candidates(score_cache, eval_df, k=10):
    ranks = []

    for row, cache_item in zip(eval_df.itertuples(index=False), score_cache):
        candidate_recipe_ids = cache_item["candidate_recipe_ids"]
        candidate_item_indices = np.array([
            item_id_map[str(recipe_id)]
            for recipe_id in candidate_recipe_ids
        ])

        user_index = user_id_map[str(row.user_id)]

        scores = lightfm_model.predict(
            user_ids=np.repeat(user_index, len(candidate_item_indices)),
            item_ids=candidate_item_indices,
            item_features=item_features_matrix,
            num_threads=4,
        )

        ranking = np.argsort(-scores, kind="stable")
        actual_position = np.where(
            candidate_recipe_ids == row.recipe_id
        )[0][0]
        ranks.append(np.where(ranking == actual_position)[0][0] + 1)

    return calculate_ranking_metrics(ranks, k=k)


lightfm_comparison_results = evaluate_lightfm_on_cached_candidates(
    score_cache=test_score_cache,
    eval_df=test_interactions,
    k=10,
)

final_results = pd.concat(
    [
        final_results,
        pd.DataFrame([
            {
                "Model": "LightFM Hybrid",
                **lightfm_comparison_results,
            }
        ]),
    ],
    ignore_index=True,
)

display(final_results.round(4))


## 9. Model Comparison

All models below are compared using the same 100-candidate sampled-ranking evaluation. Higher values are better.

In [ ]:
# Four-model comparison under the same sampled-candidate evaluation protocol
model_order = [
    "Popularity Baseline",
    "Content-based Baseline",
    "LightFM Hybrid",
]

# Keep the validation-selected weighted hybrid as the final row.
weighted_hybrid_name = final_results.loc[
    final_results["Model"].str.startswith("Hybrid"), "Model"
].iloc[0]
model_order.append(weighted_hybrid_name)

comparison_display = (
    final_results
    .set_index("Model")
    .loc[model_order]
    .reset_index()
    .copy()
)

comparison_display["Model"] = comparison_display["Model"].replace({
    weighted_hybrid_name: "Weighted Hybrid (selected)"
})

display(comparison_display.round(4))

colors = {
    "Popularity Baseline": "#4C78A8",
    "Content-based Baseline": "#F28E2B",
    "LightFM Hybrid": "#8E6C8A",
    "Weighted Hybrid (selected)": "#54A24B",
}

plot_df = comparison_display.copy()
plot_df["color"] = plot_df["Model"].map(colors)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)
metrics = ["Recall@10", "NDCG@10"]

for ax, metric in zip(axes, metrics):
    bars = ax.barh(
        plot_df["Model"],
        plot_df[metric],
        color=plot_df["color"],
        height=0.6,
    )

    ax.set_title(metric, fontsize=14, weight="bold", pad=12)
    ax.set_xlabel("Score")
    ax.set_xlim(0, min(1, plot_df[metric].max() + 0.09))
    ax.grid(axis="x", alpha=0.25)
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)

    for bar, value in zip(bars, plot_df[metric]):
        ax.text(
            bar.get_width() + 0.006,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.3f}",
            va="center",
            fontsize=11,
            weight="bold",
        )

axes[0].invert_yaxis()
fig.suptitle(
    "Recommendation Model Comparison",
    fontsize=18,
    weight="bold",
    y=1.02,
)
fig.text(
    0.5,
    -0.02,
    "Evaluation: 1 held-out liked recipe + 99 unseen candidate recipes per user (n = 10,404)",
    ha="center",
    color="#555555",
)
plt.tight_layout()
plt.show()


## 10. Conclusion

The popularity baseline is strong because the interaction data is sparse and highly skewed toward high ratings. However, the weighted hybrid combines global popularity with individual content preferences and is selected by validation before final testing.

This project demonstrates an end-to-end recommendation workflow: data cleaning, leakage-aware time splitting, baseline construction, personalized ranking, validation-based model selection, and transparent model comparison.